# 📈 Stock / Commodity Forecast

Questo notebook scarica una serie temporale da Yahoo Finance e produce previsioni con:
- **Chronos** (Amazon, zero-shot transformer)
- **N-HiTS** (NeuralForecast, training on-the-fly)

---
### Ticker di esempio
| Asset | Ticker |
|---|---|
| Apple | `AAPL` |
| S&P 500 | `^GSPC` |
| Oro | `GC=F` |
| Petrolio WTI | `CL=F` |
| Bitcoin | `BTC-USD` |
| EUR/USD | `EURUSD=X` |

In [ ]:
import sys

sys.path.insert(0, "../src")

from stock_forecast.data import download_series, get_ticker_info, PRESET_TICKERS
from stock_forecast.models import run_chronos, run_nhits, run_all
from stock_forecast.plot import plot_forecast, print_summary

## 1 · Configurazione

In [ ]:
# ── Parametri principali ──────────────────────────────────────────────
TICKER = "CL=F"  # Yahoo Finance ticker   (es: AAPL, ^GSPC, GC=F, BTC-USD)
PERIOD = "max"  # storico scaricato       (6mo | 1y | 2y | 5y | 10y | max)
INTERVAL = "1d"  # granularità             (1d | 1wk | 1mo)
HORIZON = 30  # giorni / barre da prevedere

# ── Opzioni modelli ───────────────────────────────────────────────────
CHRONOS_SIZE = "large"  # tiny | mini | small | base | large
NHITS_STEPS = 50  # iterazioni di training N-HiTS
RUN_BOTH = True  # False → solo Chronos (più veloce)

print(f"Ticker: {TICKER}  |  Periodo: {PERIOD}  |  Orizzonte: {HORIZON} barre")

## 2 · Download dati

In [ ]:
info = get_ticker_info(TICKER)
print(f"\nAsset   : {info.get('name', TICKER)}")
print(f"Ticker  : {info['ticker']}")
print(f"Currency: {info.get('currency', '—')}")
print(f"Sector  : {info.get('sector', '—')}")

df = download_series(TICKER, period=PERIOD, interval=INTERVAL)
print(f"\nSerie scaricata: {len(df)} osservazioni")
print(f"Da {df['ds'].min().date()}  →  {df['ds'].max().date()}")
df.tail()

## 3 · Previsione

In [ ]:
if RUN_BOTH:
    results = run_all(df, horizon=HORIZON, chronos_size=CHRONOS_SIZE, nhits_steps=NHITS_STEPS)
else:
    r = run_chronos(df, horizon=HORIZON, model_size=CHRONOS_SIZE)
    results = {r.model_name: r}

print_summary(results, ticker_info=info)

## 4 · Grafico interattivo

In [ ]:
fig = plot_forecast(
    results,
    ticker=info.get("name", TICKER),
    last_n_days=100,
    show_volume=True,
)
fig.show()

## 5 · Salva grafico

In [ ]:
import os

os.makedirs("../outputs", exist_ok=True)
safe_ticker = TICKER.replace("=", "_").replace("^", "")
out_path = f"../outputs/{safe_ticker}_forecast.html"
fig.write_html(out_path)
print(f"Grafico salvato in: {out_path}")